# Miller Capstone Project

**Author:** Dan Miller

**Date:** July 6th, 2026

**Objective:** Successfully create a ML model to accurately predict whether a movie will be successful before release from the Kaggle TMDB 5000 Movie Dataset.

## Introduction

This project explores the Kaggle TMDB 5000 Movie Dataset.  The dataset contains both numerical and categorical data that describe the metadata of movies, and we want to predict, based off of certain input features, if a movie will be successful.  After exploring the dataset, input features will be chosen and ML models will be made.  A comparison of the models' performances will be shown and discussed.

## Imports

In [1318]:
# Imports

import matplotlib.pyplot as plt
import json
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, LabelEncoder, OneHotEncoder, MultiLabelBinarizer

## Section 1. Import and Inspect the Data

### 1.1 Load the Data

In [1319]:
file_path1 = "data/tmdb_5000_movies.csv"

file_path2 = "data/tmdb_5000_credits.csv"

# Load the datasets
movies = pd.read_csv(file_path1)
credits = pd.read_csv(file_path2)

# Combine the datasets on the 'id' column

df = movies.merge(credits, left_on='id', right_on='movie_id', how='inner')
print(df.shape)
print(df.columns)

# Display the first few rows
df.head()

(4803, 24)
Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title_x', 'vote_average',
       'vote_count', 'movie_id', 'title_y', 'cast', 'crew'],
      dtype='str')


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,spoken_languages,status,tagline,title_x,vote_average,vote_count,movie_id,title_y,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


### 1.2 Display Summary Statistics

In [1320]:
# Display the data types
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [1321]:
# Display summary statistics
print(df.describe())

             budget             id   popularity       revenue      runtime  \
count  4.803000e+03    4803.000000  4803.000000  4.803000e+03  4801.000000   
mean   2.904504e+07   57165.484281    21.492301  8.226064e+07   106.875859   
std    4.072239e+07   88694.614033    31.816650  1.628571e+08    22.611935   
min    0.000000e+00       5.000000     0.000000  0.000000e+00     0.000000   
25%    7.900000e+05    9014.500000     4.668070  0.000000e+00    94.000000   
50%    1.500000e+07   14629.000000    12.921594  1.917000e+07   103.000000   
75%    4.000000e+07   58610.500000    28.313505  9.291719e+07   118.000000   
max    3.800000e+08  459488.000000   875.581305  2.787965e+09   338.000000   

       vote_average    vote_count       movie_id  
count   4803.000000   4803.000000    4803.000000  
mean       6.092172    690.217989   57165.484281  
std        1.194612   1234.585891   88694.614033  
min        0.000000      0.000000       5.000000  
25%        5.600000     54.000000    9014.

## Section 2. Data Exploration & Preparation

### 2.1 Handle Missing  & Duplicate Values

In [1322]:
# Identify missing values
print(df.isnull().sum())

budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title_x                    0
vote_average               0
vote_count                 0
movie_id                   0
title_y                    0
cast                       0
crew                       0
dtype: int64


In [1323]:
# Remove duplicates
df = df.drop_duplicates()

# Print number of budget and revenue values that are 0
print("Number of budget values that are 0:", (df['budget'] == 0).sum())
print("Number of revenue values that are 0:", (df['revenue'] == 0).sum())

Number of budget values that are 0: 1037
Number of revenue values that are 0: 1427


In [1324]:
# Replace budget and revnue values of 0 with NaN
df['budget'] = df['budget'].replace(0, pd.NA)
df['revenue'] = df['revenue'].replace(0, pd.NA)

# Drop rows with missing values in the 'budget' and 'revenue' columns
df = df.dropna(subset=['budget', 'revenue'])

# Drop homepage and tagline columns
df = df.drop(columns=['homepage', 'tagline'])

# Drop rows with missing values in the 'release_date' and 'overview' columns
df = df.dropna(subset=['release_date', 'overview'])

# Fill missing values in the 'runtime' column with the median runtime
median_runtime = df['runtime'].median()
df['runtime'] = df['runtime'].fillna(median_runtime)

# Print the number of missing values in each column after cleaning
print(df.isnull().sum())

budget                  0
genres                  0
id                      0
keywords                0
original_language       0
original_title          0
overview                0
popularity              0
production_companies    0
production_countries    0
release_date            0
revenue                 0
runtime                 0
spoken_languages        0
status                  0
title_x                 0
vote_average            0
vote_count              0
movie_id                0
title_y                 0
cast                    0
crew                    0
dtype: int64


### 2.2 Clean Data

In [1325]:
# Drop unnecessary columns
columns_to_drop = ['movie_id', 'status', 'title_x', 'title_y', 'vote_average', 'vote_count', 'popularity', 'spoken_languages', 'overview', 'production_countries', 'id', 'original_language']
df = df.drop(columns=columns_to_drop)

#Rename original_title to title
df = df.rename(columns={'original_title': 'title'})

# Convert release_date to datetime
df['release_date'] = pd.to_datetime(df['release_date'])


In [1326]:
# Convert budget and revenue back to numeric
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce')

# Create ROI column
df['roi'] = df['revenue'] / df['budget']

# Define success based on ROI threshold
df['success'] = (df['roi'] >= 2.0).astype(int)

# Drop revenue and roi columns
df = df.drop(columns=['revenue', 'roi'])

### 2.3 Feature Engineering

In [1327]:
# Log transform the budget column
df['log_budget'] = np.log1p(df['budget'])

# Extract year and month from release_date
df['release_year'] = df['release_date'].dt.year
df['release_month'] = df['release_date'].dt.month

# Drop release_date column
df = df.drop(columns=['release_date'])

# Display the first few rows of new date columns
df[['release_year', 'release_month']].head()


,release_year,release_month
0,2009,12
1,2007,5
2,2015,10
3,2012,7
4,2012,3


In [1328]:
# Parse JSON columns
def parse_names(value):
    if pd.isna(value):
        return []
    try:
        return [item["name"] for item in json.loads(value)]
    except (TypeError, json.JSONDecodeError):
        return []

json_columns = [
    'genres',
    'keywords',
    'production_companies',
]

for col in json_columns:
    df[col] = df[col].apply(parse_names)

# Display the first few rows of the parsed JSON columns
df[json_columns].head()

,genres,keywords,production_companies
0,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Ingenious Film Partners, Twentieth Century Fo..."
1,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Walt Disney Pictures, Jerry Bruckheimer Films..."
2,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Columbia Pictures, Danjaq, B24]"
3,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Legendary Pictures, Warner Bros., DC Entertai..."
4,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...",[Walt Disney Pictures]


In [1329]:
# Parse cast and crew
df['cast'] = df['cast'].apply(json.loads)
df['crew'] = df['crew'].apply(json.loads)

# Extract the top cast members
def get_top_cast(cast_list, top_n=3):
    return [person["name"] for person in cast_list[:top_n]]

df['cast'] = df['cast'].apply(get_top_cast)

# Extract directors from crew
def get_director(crew_list):
    for person in crew_list:
        if person["job"] == "Director":
            return person["name"]
    return None

df['director'] = df['crew'].apply(get_director)

# Drop the original crew column
df = df.drop(columns=['crew'])

# Display the first few rows of the cast and director columns
df[['cast', 'director']].head()

,cast,director
0,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron
1,"[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski
2,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes
3,"[Christian Bale, Michael Caine, Gary Oldman]",Christopher Nolan
4,"[Taylor Kitsch, Lynn Collins, Samantha Morton]",Andrew Stanton


In [1330]:
# Get the most common production companies
production_company_counts = Counter()

for companies in df['production_companies']:
    production_company_counts.update(companies)

common_companies = {
    company
    for company, count in production_company_counts.items()
    if count >= 10
}

df['production_companies'] = df['production_companies'].apply(
    lambda companies: [company for company in companies if company in common_companies]
)

# Get the most common actors
actor_counts = Counter()

for actor in df['cast']:
    actor_counts.update(actor)

common_actors = {
    actor
    for actor, count in actor_counts.items()
    if count >= 10
}

df['cast'] = df['cast'].apply(
    lambda actors: [actor for actor in actors if actor in common_actors]
)

# Get the most common directors

director_counts = df['director'].value_counts()

common_directors = director_counts[director_counts >= 5].index

df['director'] = df['director'].apply(
    lambda x: x if x in common_directors else "Other"
)

In [1331]:
# Apply MultiLabelBinarizer to genres, production_companies, and cast columns
mlb = MultiLabelBinarizer()

genres_df = pd.DataFrame(mlb.fit_transform(df['genres']), columns=mlb.classes_, index=df.index)
companies_df = pd.DataFrame(mlb.fit_transform(df['production_companies']), columns=mlb.classes_, index=df.index)
cast_df = pd.DataFrame(mlb.fit_transform(df['cast']), columns=mlb.classes_, index=df.index)

df = pd.concat([df, genres_df], axis=1)
df = df.drop(columns=['genres'])

df = pd.concat([df, companies_df], axis=1)
df = df.drop(columns=['production_companies'])

df = pd.concat([df, cast_df], axis=1)
df = df.drop(columns=['cast'])

In [1332]:
# One hot encode the director

director_dummies = pd.get_dummies(
    df['director'],
    prefix='director',
    dtype=int
)

df = pd.concat([df, director_dummies], axis=1)
df = df.drop(columns=['director'])

In [1333]:
# Use TF-IDF Vectorizer to encode the keywords column

df['keywords_text'] = df['keywords'].apply(
    lambda x: ' '.join(
        keyword.replace(' ', '_') for keyword in x
    )
)

tfidf = TfidfVectorizer(max_features=75)

# keyword_features = tfidf.fit_transform(df['keywords_text'])

# keywords_df = pd.DataFrame(
#     keyword_features.toarray(),
#     columns=[f"keyword_{x}" for x in tfidf.get_feature_names_out()],
#     index=df.index
# )

#df = pd.concat([df, keywords_df], axis=1)
# df = df.drop(columns=['keywords', 'keywords_text'])

#df.head()

## Section 3. Feature Selection and Justification

### 3.1 Choose Features and Target

The input features are as follows:
- log_budget
- runtime
- release_year
- release_month
- genres
- production_companies
- cast
- director
- keywords

The target feature is **success**

### 3.2 Define X and y

In [1334]:
X = df.drop(columns=['budget', 'title', 'success'])
y = df['success']

## Section 4. Train a Model

### 4.1 Split the Data

In [1335]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_keywords = tfidf.fit_transform(X_train['keywords_text'])

test_keywords = tfidf.transform(X_test['keywords_text'])

# Convert TF-IDF into columns

keyword_columns = [
    f"keyword_{word}"
    for word in tfidf.get_feature_names_out()
]

train_keywords_df = pd.DataFrame(
    train_keywords.toarray(),
    columns=keyword_columns,
    index=X_train.index
)

test_keywords_df = pd.DataFrame(
    test_keywords.toarray(),
    columns=keyword_columns,
    index=X_test.index
)

# Add IT-IDF features back

X_train = X_train.drop(columns=['keywords', 'keywords_text'])
X_test = X_test.drop(columns=['keywords', 'keywords_text'])

X_train = pd.concat([X_train, train_keywords_df], axis=1)
X_test = pd.concat([X_test, test_keywords_df], axis=1)